In [ ]:
import pandas as pd
%load_ext autoreload
%autoreload 2

from pathlib import Path
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np
import napari
import colorcet as cc

import dnt

spots_directory = Path(r"C:\Tracking\BlastodermAnalysis\data\spots")
embryo_overview = pd.read_excel(spots_directory / "overview.xlsx", sheet_name="Sheet1")
save_path = Path(r"C:\Tracking\BlastodermAnalysis\figures\figure_2")


embryo_overview = embryo_overview[embryo_overview["good"]]
included = embryo_overview["Embryo"].astype(str).tolist()
condition_map = {
    str(embryo): condition for embryo, condition in zip(embryo_overview["Embryo"], embryo_overview["condition"])
}
print(condition_map)

dnt.set_plot_style()
spots_dfs, stems = dnt.load_spots_data(spots_directory, included)

print(stems)

df = spots_dfs[0]
cycles = [10, 11, 12, 13, 14]

print(df.columns)

greens = ["#143601","#1a4301","#245501","#538d22","#73a942","#aad576"][::-1]
blues = ["#012a4a","#01497c","#2a6f97","#468faf","#89c2d9"][::-1]
reds = ["#641220","#85182a","#a71e34","#bd1f36", "#da1e37"][::-1]
oranges = ["#fbba72","#ca5310","#bb4d00","#8f250c","#691e06"]
condition_pal_map = {
    "wt": blues,
    "bcd": reds,
    "trk": greens,
}
condition_main_colors = {
    c: cmap[2] for c, cmap in condition_pal_map.items()
}

In [ ]:
from scipy.spatial import KDTree
from itertools import pairwise
from collections import defaultdict

displacements = defaultdict(list)

for left, right in pairwise([0.0, 0.2, 0.4, 0.6, 0.8, 1.0]):
    for k, df in enumerate(spots_dfs):
        df = df[df["AP"].between(left, right)].copy()
        df["child_id"] = df.index.map({
            parent: child for parent, child in zip(df["parent_id"], df.index)
        })
        for metric in ["dz", "dy", "dx"]:
            df[f"child_{metric}"] = df["child_id"].map(df[metric])
        cycle = 11
        frame_start = int(df[df["cycle"] == cycle].groupby("tracklet_id")["frame"].min().quantile(0.5))
        frame_end = int(df[df["cycle"] == cycle].groupby("tracklet_id")["frame"].max().quantile(0.5))

        dis, dx = [], []

        for frame in range(frame_start+5, frame_end-5):
            frame_df = df[df["frame"] == frame].copy()
            points = frame_df[["z", "y", "x"]].values

            changes = frame_df[["child_dz", "child_dy", "child_dx"]].values
            tree = KDTree(points)
            dists, ii = tree.query(points, k=2)
            dis.extend(dists[:, 1:].flatten())

            a_index = np.arange(ii.shape[0])[:, None]

            relative_position_vectors = points[:, None, :] - points[ii[:, 1:], :]
            relative_movement_vectors = changes[:, None, :] - changes[ii[:, 1:], :]

            normalized_relative_position_vectors = relative_position_vectors / np.linalg.norm(relative_position_vectors, axis=-1, keepdims=True)
            dx_values = np.einsum("ijk, ijk -> ij", normalized_relative_position_vectors, relative_movement_vectors)
            dx.extend(dx_values.flatten())

        displacements["location"].append(left)
        displacements["distance"].append(np.nanmean(dis))
        displacements["dx"].append(np.nanmean(dx))
        displacements["source"].append(k)
        condition = condition_map[stems[k][:-6]]
        displacements["condition"].append(condition)
        # plt.scatter(dis, dx, s=0.3, alpha=0.5, color="r")
        # sns.lineplot(x=(np.array(dis) // 0.5) * 0.5 , y=dx, errorbar=None, label=left)
displacements_df = pd.DataFrame(displacements)


In [ ]:
sns.barplot(
    data=displacements_df[displacements_df["condition"].isin(["wt", "bcd"])], x="location", y="dx", hue="condition",
    palette=condition_main_colors, errorbar=None)
sns.stripplot(
    data=displacements_df[displacements_df["condition"].isin(["wt", "bcd"])], x="location", y="dx", hue="condition",
    palette="dark:k", dodge=True, alpha=0.5, size=5)

plt.show()

sns.lineplot(
    data=displacements_df[displacements_df["condition"].isin(["wt", "bcd"])], x="location", y="distance", hue="condition", style="source", dashes=False,
    palette=condition_main_colors, errorbar=None, legend=False)
plt.show()